# 04 — Player-Manager Compatibility, Explainability & Club-Bias Check

**CRISP-DM phase:** Modelling / Evaluation.

Implements:
1. Player -> manager compatibility as cosine similarity between the player's tactical-dimension vector and each manager's centroid (brief SS23 -- explicitly a similarity score, not a probability).
2. Football-language explanations decomposed from the actual per-dimension contributions (brief SS24-26).
3. A club-identity-bias check (brief SS18-19, SS32) using De Zerbi's four in-scope clubs -- the only manager in our sample with more than one club.

**Data-coverage note (read before trusting any number below):** a systematic coverage audit (`outputs/audit/data_availability_matrix.md` SS5, `decision_log.md` D-006) found that FBref's match-level 'passing', 'passing_types', 'possession' and 'keeper' blocks are simply not populated for Serie A/Ligue 1 before the 2023/24 season -- so Sassuolo (2018/19-2020/21) and Benevento (2017/18) have **no data at all** for 5 of our 7 tactical dimensions (only pressing_intensity and individual_defending survive, the latter partially, since they come from the always-available 'defense'/'misc' blocks). This is now handled correctly (missing = NaN, propagated honestly, never silently treated as 0/average -- see `outputs/results/task4_evaluation_findings.md` for the full incident writeup), but it means any player-manager comparison touching Sassuolo or Benevento is thin -- `n_dims_used` and `low_confidence` flags on every row exist specifically so this isn't hidden.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd

from data_sources import DATA_PROCESSED
from football_features import DIMENSION_WEIGHTS, build_club_within_manager_profiles
from compatibility import (
    compute_compatibility, explain_player_manager_fit, rescale_to_index,
    DIMENSION_LABELS, masked_cosine_similarity_matrix,
)

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

player_profiles = pd.read_parquet(DATA_PROCESSED / "player_profiles.parquet")
manager_profiles = pd.read_parquet(DATA_PROCESSED / "manager_profiles.parquet")
dim_cols = list(DIMENSION_WEIGHTS.keys())
manager_profiles.set_index("manager").round(3)

,TD1_buildup_directness,TD3_progression_passing,TD4_progression_carrying,TD6_chance_creation,TD7_box_presence,TD9_pressing_intensity,TD11_individual_defending
manager,,,,,,,
Jürgen Klopp,0.318,0.237,-0.012,-0.035,0.079,0.229,-0.139
Pep Guardiola,0.446,0.293,0.102,-0.037,0.056,0.304,-0.182
Roberto De Zerbi,0.251,0.139,0.015,-0.017,-0.022,-0.001,-0.163


## 1. Compute compatibility for every player-season against every manager

In [2]:
compat = compute_compatibility(player_profiles, manager_profiles, dim_cols, min_dims_required=4)
compat["compatibility_index_0_100"] = compat.groupby("scored_against_manager")["compatibility_similarity"].transform(rescale_to_index)
print(compat.shape)
print(f"low_confidence (fewer than 4 of 7 dimensions available): {compat.low_confidence.mean():.1%} of rows")
compat.groupby("team")["n_dims_used"].mean().round(2)

(7890, 18)
low_confidence (fewer than 4 of 7 dimensions available): 6.6% of rows


team
Benevento                 2.00
Brighton & Hove Albion    6.96
Liverpool                 6.88
Manchester City           6.93
Marseille                 6.90
Sassuolo                  6.90
Name: n_dims_used, dtype: float64

## 2. Best-fit manager per player-season, restricted to adequately-covered comparisons

Excludes `low_confidence` rows (fewer than 4 of 7 dimensions available) from the headline statistic -- including them would let a handful of measurable dimensions (often just pressing intensity) decide a "best fit" that the data cannot actually support. This mostly affects Sassuolo/Benevento player-seasons, per the coverage note above.

In [3]:
confident = compat[~compat.low_confidence]
print(f"{len(confident)} of {len(compat)} rows retained ({len(confident)/len(compat):.1%})")

best_fit = confident.loc[confident.groupby(["Player", "Season_End_Year", "team"])["compatibility_similarity"].idxmax()]
match_rate = (best_fit["scored_against_manager"] == best_fit["actual_manager"]).mean()
print(f"Share of (adequately-covered) player-seasons whose single best-fit manager IS the manager they actually played under: {match_rate:.1%}  (n={len(best_fit)}, chance baseline ~33%)")
pd.crosstab(best_fit["actual_manager"], best_fit["scored_against_manager"])

7368 of 7890 rows retained (93.4%)
Share of (adequately-covered) player-seasons whose single best-fit manager IS the manager they actually played under: 42.1%  (n=561, chance baseline ~33%)


scored_against_manager,Jürgen Klopp,Pep Guardiola,Roberto De Zerbi
actual_manager,,,
Jürgen Klopp,66,44,78
Pep Guardiola,46,47,77
Roberto De Zerbi,34,46,123


## 2b. Is the best-fit match rate statistically significant?

SS2 reports a single point estimate (42.1%, n=561). On its own, a point estimate does not say how much of that number is noise from the particular 561 player-seasons in the sample, nor how unlikely a rate this high would be if the model's best-fit decisions had no real relationship to the manager a player actually played under. Two checks close that gap, both using only the already-computed `best_fit` table above, with no new data or assumptions:

1. **Bootstrap confidence interval.** Resample the 561 player-seasons with replacement 5000 times and recompute the match rate each time. The spread of these 5000 rates is a direct, assumption-light estimate of how much the 42.1% figure would plausibly move under a different sample of the same size drawn from the same underlying population.
2. **Permutation test.** Shuffle the true `actual_manager` label among the 561 rows 5000 times while leaving the model's own `scored_against_manager` decision untouched, and recompute the match rate each time. This builds an empirical null distribution for "how often would a best-fit decision line up with the true manager, if there were no real association between them, given this sample's actual manager mix" -- a stricter and more precise baseline than the naive 1-in-3 chance figure used elsewhere in this project, because it is derived from the sample's real class balance (De Zerbi, Klopp and Guardiola are not represented in exactly equal numbers) rather than assumed.

In [4]:
boot_rng = np.random.default_rng(0)
match_col = (best_fit["scored_against_manager"] == best_fit["actual_manager"]).to_numpy()
n_bf = len(best_fit)

B = 5000
boot_rates = np.array([match_col[boot_rng.integers(0, n_bf, size=n_bf)].mean() for _ in range(B)])
ci_low, ci_med, ci_high = np.percentile(boot_rates, [2.5, 50, 97.5])
print(f"Bootstrap (B={B}) 95% CI for the best-fit match rate: [{ci_low:.1%}, {ci_high:.1%}]  (median {ci_med:.1%}, SE={boot_rates.std():.4f})")

P = 5000
actual = best_fit["actual_manager"].to_numpy()
scored = best_fit["scored_against_manager"].to_numpy()
perm_rates = np.array([(boot_rng.permutation(actual) == scored).mean() for _ in range(P)])
empirical_chance = perm_rates.mean()
p_value = (perm_rates >= match_rate).mean()
print(f"\nPermutation test (P={P}):")
print(f"  Empirical chance baseline (mean of null distribution): {empirical_chance:.1%}  (null std={perm_rates.std():.4f})")
print(f"  Observed rate: {match_rate:.1%}")
print(f"  p-value, P[null >= observed]: {p_value:.5f}" if p_value > 0 else f"  p-value: < {1/P:.5f} (no permutation reached the observed rate)")

significance_summary = pd.Series({
    "observed_rate": match_rate, "n": n_bf,
    "bootstrap_ci_low": ci_low, "bootstrap_ci_high": ci_high, "bootstrap_se": boot_rates.std(),
    "empirical_chance_baseline": empirical_chance, "p_value": p_value,
})
significance_summary

Bootstrap (B=5000) 95% CI for the best-fit match rate: [37.8%, 46.2%]  (median 42.1%, SE=0.0210)

Permutation test (P=5000):
  Empirical chance baseline (mean of null distribution): 34.0%  (null std=0.0196)
  Observed rate: 42.1%
  p-value, P[null >= observed]: 0.00020


observed_rate                  0.420677
n                            561.000000
bootstrap_ci_low               0.377897
bootstrap_ci_high              0.461676
bootstrap_se                   0.021030
empirical_chance_baseline      0.340324
p_value                        0.000200
dtype: float64

## 3. Worked example: the brief's own success-criterion output (SS51)

In [5]:
def report_for(player_name, season, team):
    prow = player_profiles[(player_profiles.Player == player_name) & (player_profiles.Season_End_Year == season) & (player_profiles.team == team)].iloc[0]
    sub = compat[(compat.Player == player_name) & (compat.Season_End_Year == season) & (compat.team == team)].sort_values("compatibility_similarity", ascending=False)
    print(f"PLAYER: {player_name}   POSITION: {prow.Pos}   SEASON: {season}   TEAM: {team}   (actually played under: {prow.manager})\n")
    print("MANAGER FIT (cosine similarity -- NOT a probability; rescaled 0-100 index for readability)")
    for _, r in sub.iterrows():
        flag = "  [LOW CONFIDENCE]" if r.low_confidence else ""
        print(f"  {r.scored_against_manager:<18s} similarity={r.compatibility_similarity:+.3f}   index={r.compatibility_index_0_100:5.1f}   dims_used={r.n_dims_used}/7{flag}")
    best = sub.iloc[0]
    print(f"\nBEST FIT: {best.scored_against_manager.upper()}\n")
    expl = explain_player_manager_fit(best, prow[dim_cols], manager_profiles.set_index("manager").loc[best.scored_against_manager], dim_cols)
    print("Why?")
    for s in expl["data_derived_evidence"]:
        print(f"  + {s}")
    print("Limiting factors:")
    for s in expl["limiting_factors"]:
        print(f"  - {s}")
    print("\n" + "="*100 + "\n")
    return sub

examples = player_profiles.sort_values("total_minutes", ascending=False).groupby("manager").head(1)
for _, ex in examples.iterrows():
    report_for(ex.Player, ex.Season_End_Year, ex.team)

PLAYER: Virgil van Dijk   POSITION: CB   SEASON: 2020   TEAM: Liverpool   (actually played under: Jürgen Klopp)

MANAGER FIT (cosine similarity -- NOT a probability; rescaled 0-100 index for readability)
  Roberto De Zerbi   similarity=+0.842   index= 92.1   dims_used=7/7
  Jürgen Klopp       similarity=+0.679   index= 84.0   dims_used=7/7
  Pep Guardiola      similarity=+0.629   index= 81.5   dims_used=7/7
  Roberto De Zerbi   similarity=+0.617   index= 80.8   dims_used=7/7
  Jürgen Klopp       similarity=+0.466   index= 73.3   dims_used=7/7
  Pep Guardiola      similarity=+0.383   index= 69.2   dims_used=7/7

BEST FIT: ROBERTO DE ZERBI

Why?
  + build-up / possession-based play: player z-score +0.31 (above the league average for the position), contributing +0.078 to the similarity with Roberto De Zerbi.
  + individual defending / duels: player z-score -0.12 (below the league average for the position), contributing +0.019 to the similarity with Roberto De Zerbi.
  + progressive passin

## 4. Club-identity-bias check (brief SS18-19)

De Zerbi's four in-scope clubs, compared pairwise. Similarity is masked to the dimensions both club profiles actually have data for (`masked_cosine_similarity_matrix`) -- Benevento and Sassuolo only have `pressing_intensity` and (partially) `individual_defending` populated, per the coverage note above, so comparisons involving them use far fewer dimensions than comparisons among Liverpool/Man City/Brighton/Marseille. `n_dims_used` is reported alongside every similarity so this is visible, not hidden.

In [6]:
club_profiles = build_club_within_manager_profiles(player_profiles, dim_cols)
club_profiles.round(3)

,manager,team,TD1_buildup_directness,TD3_progression_passing,TD4_progression_carrying,TD6_chance_creation,TD7_box_presence,TD9_pressing_intensity,TD11_individual_defending
0,Jürgen Klopp,Liverpool,0.318,0.237,-0.012,-0.035,0.079,0.229,-0.139
1,Pep Guardiola,Manchester City,0.446,0.293,0.102,-0.037,0.056,0.304,-0.182
2,Roberto De Zerbi,Benevento,NaN,NaN,NaN,NaN,NaN,-0.093,-0.148
3,Roberto De Zerbi,Brighton & Hove Albion,0.199,0.129,-0.018,0.022,0.075,0.038,-0.109
4,Roberto De Zerbi,Marseille,0.213,0.135,-0.028,-0.017,0.000,-0.015,0.010
5,Roberto De Zerbi,Sassuolo,0.291,0.146,0.044,-0.041,-0.086,0.001,-0.236


In [7]:
labels = club_profiles["manager"] + " @ " + club_profiles["team"]
vecs = club_profiles[dim_cols].to_numpy(dtype=float)
sim, ndims = masked_cosine_similarity_matrix(vecs)
sim_matrix = pd.DataFrame(sim, index=labels, columns=labels)
ndims_matrix = pd.DataFrame(ndims, index=labels, columns=labels)
print("Similarity:")
display(sim_matrix.round(3))
print("\nDimensions used per pair (out of 7):")
display(ndims_matrix)

Similarity:


,Jürgen Klopp @ Liverpool,Pep Guardiola @ Manchester City,Roberto De Zerbi @ Benevento,Roberto De Zerbi @ Brighton & Hove Albion,Roberto De Zerbi @ Marseille,Roberto De Zerbi @ Sassuolo
Jürgen Klopp @ Liverpool,1.000,0.979,-0.017,0.917,0.773,0.764
Pep Guardiola @ Manchester City,0.979,1.000,-0.022,0.887,0.757,0.801
Roberto De Zerbi @ Benevento,-0.017,-0.022,1.000,0.626,-0.026,0.843
Roberto De Zerbi @ Brighton & Hove Albion,0.917,0.887,0.626,1.000,0.830,0.823
Roberto De Zerbi @ Marseille,0.773,0.757,-0.026,0.830,1.000,0.743
Roberto De Zerbi @ Sassuolo,0.764,0.801,0.843,0.823,0.743,1.000



Dimensions used per pair (out of 7):


,Jürgen Klopp @ Liverpool,Pep Guardiola @ Manchester City,Roberto De Zerbi @ Benevento,Roberto De Zerbi @ Brighton & Hove Albion,Roberto De Zerbi @ Marseille,Roberto De Zerbi @ Sassuolo
Jürgen Klopp @ Liverpool,7,7,2,7,7,7
Pep Guardiola @ Manchester City,7,7,2,7,7,7
Roberto De Zerbi @ Benevento,2,2,2,2,2,2
Roberto De Zerbi @ Brighton & Hove Albion,7,7,2,7,7,7
Roberto De Zerbi @ Marseille,7,7,2,7,7,7
Roberto De Zerbi @ Sassuolo,7,7,2,7,7,7


In [8]:
dz_rows = [i for i, l in enumerate(labels) if l.startswith("Roberto De Zerbi")]
other_rows = [i for i in range(len(labels)) if i not in dz_rows]

# Restrict the headline comparison to pairs backed by at least 4 of 7 dimensions,
# for the same reason as SS2 above -- a 1-2 dimension comparison is too thin to
# support a claim either way.
def well_covered_pairs(rows_a, rows_b, min_dims=4):
    vals = []
    for i in rows_a:
        for j in rows_b:
            if i != j and ndims[i, j] >= min_dims:
                vals.append(sim[i, j])
    return vals

within_dz = well_covered_pairs(dz_rows, dz_rows)
dz_to_others = well_covered_pairs(dz_rows, other_rows)

print(f"De Zerbi club-to-club similarity, well-covered pairs only: mean={np.mean(within_dz):.3f}  (n={len(within_dz)} pairs)" if within_dz else "De Zerbi club-to-club: no pair reaches the 4-dimension coverage threshold.")
print(f"De Zerbi clubs vs. Guardiola/Klopp, well-covered pairs only: mean={np.mean(dz_to_others):.3f}  (n={len(dz_to_others)} pairs)" if dz_to_others else "De Zerbi vs others: no pair reaches the 4-dimension coverage threshold.")
print()
if within_dz and dz_to_others:
    if np.mean(within_dz) > np.mean(dz_to_others):
        print("RESULT: among comparisons with adequate dimensional coverage, De Zerbi's own clubs resemble each other MORE than they resemble the other two managers.")
        print("This is evidence the model captures a signal that survives a change of squad/league, not just club identity.")
    else:
        print("RESULT: even restricted to well-covered comparisons, De Zerbi's clubs do not clearly resemble each other more than they resemble the other managers.")
        print("Report this as a genuine limitation, not a result to hide -- see outputs/results/task4_evaluation_findings.md.")
else:
    print("RESULT: insufficient well-covered pairs to run this check at all -- Sassuolo/Benevento's missing dimensions leave too few comparisons above the coverage threshold.")
    print("The only genuinely well-covered De Zerbi comparison is Brighton vs Marseille; see the raw similarity matrix above for that specific pair.")

De Zerbi club-to-club similarity, well-covered pairs only: mean=0.799  (n=6 pairs)
De Zerbi clubs vs. Guardiola/Klopp, well-covered pairs only: mean=0.816  (n=6 pairs)

RESULT: even restricted to well-covered comparisons, De Zerbi's clubs do not clearly resemble each other more than they resemble the other managers.
Report this as a genuine limitation, not a result to hide -- see outputs/results/task4_evaluation_findings.md.


## 5. Save outputs

In [9]:
compat.to_parquet(DATA_PROCESSED / "player_manager_compatibility.parquet", index=False)
club_profiles.to_csv(Path.cwd().parent / "outputs" / "tables" / "club_within_manager_profiles.csv", index=False)
sim_matrix.to_csv(Path.cwd().parent / "outputs" / "tables" / "club_bias_similarity_matrix.csv")
ndims_matrix.to_csv(Path.cwd().parent / "outputs" / "tables" / "club_bias_ndims_matrix.csv")
print("saved")

saved


## 6. Dimension importance via permutation (explainability polish)

SHAP requires a trained predictive model; our compatibility score is a similarity computation, not a fitted classifier, so SHAP does not apply directly. The natural equivalent here is **permutation importance on the best-fit task**: shuffle one tactical dimension across all player-seasons (breaking its link to the player while leaving every other dimension intact), recompute the best-fit manager for the well-covered sample, and measure how much the match rate against the actual manager degrades. A dimension whose shuffling hurts accuracy the most is the one doing the most discriminating work -- the same logic as SHAP/permutation importance for a trained model, adapted to a similarity-based method, and documented here as a deliberate substitution rather than a silent one.

In [10]:
import numpy as np
from compatibility import compute_compatibility

rng = np.random.default_rng(0)

def best_fit_match_rate(pp, mp, dims, min_dims=4):
    c = compute_compatibility(pp, mp, dims, min_dims_required=min_dims)
    conf = c[~c.low_confidence]
    bf = conf.loc[conf.groupby(["Player", "Season_End_Year", "team"])["compatibility_similarity"].idxmax()]
    return (bf["scored_against_manager"] == bf["actual_manager"]).mean(), len(bf)

baseline_rate, baseline_n = best_fit_match_rate(player_profiles, manager_profiles, dim_cols)
print(f"Baseline match rate: {baseline_rate:.1%} (n={baseline_n})\n")

importances = {}
for dim in dim_cols:
    pp_shuffled = player_profiles.copy()
    pp_shuffled[dim] = rng.permutation(pp_shuffled[dim].values)
    rate, n = best_fit_match_rate(pp_shuffled, manager_profiles, dim_cols)
    importances[dim] = baseline_rate - rate
    print(f"  shuffle {dim:<28s} -> match rate {rate:.1%}  (drop of {baseline_rate-rate:+.1%})")

importance_series = pd.Series(importances).sort_values(ascending=False)
print("\nRanked by importance (larger drop = more discriminating):")
print(importance_series.round(4))

Baseline match rate: 42.1% (n=561)



  shuffle TD1_buildup_directness       -> match rate 38.9%  (drop of +3.2%)


  shuffle TD3_progression_passing      -> match rate 38.9%  (drop of +3.2%)


  shuffle TD4_progression_carrying     -> match rate 39.4%  (drop of +2.7%)


  shuffle TD6_chance_creation          -> match rate 42.8%  (drop of -0.7%)


  shuffle TD7_box_presence             -> match rate 42.4%  (drop of -0.4%)


  shuffle TD9_pressing_intensity       -> match rate 34.6%  (drop of +7.5%)


  shuffle TD11_individual_defending    -> match rate 40.3%  (drop of +1.8%)

Ranked by importance (larger drop = more discriminating):
TD9_pressing_intensity       0.0749
TD1_buildup_directness       0.0321
TD3_progression_passing      0.0321
TD4_progression_carrying     0.0267
TD11_individual_defending    0.0178
TD7_box_presence            -0.0036
TD6_chance_creation         -0.0071
dtype: float64


**Reading this table.** A dimension with importance near zero (or negative -- shuffling it *improves* the match rate) is not contributing useful discriminating signal in the current pipeline; this is expected for at least one or two dimensions given the SS5-6 findings above, and should be reported as such rather than reading noise as a real ranking. Save the ranking for the report's interpretability section.

In [11]:
importance_series.to_csv(Path.cwd().parent / "outputs" / "tables" / "dimension_permutation_importance.csv", header=["importance"])
print("saved")

saved


## 7. Team formation profiles: an unused signal, added descriptively

The recommended next steps written into this project's report (section 6.4) flagged the starting formation recorded for every match, `Home_Formation` / `Away_Formation`, collected as part of task #2 back at the start of the project but never analysed. This section closes that gap.

**A genuine complication was found and worked around while doing this.** The plan was to fold formation into the compatibility score as an eighth tactical dimension, the same way every other dimension is built. Re-downloading the match-level source to confirm coverage showed that `worldfootballR_data`'s `fb_advanced_match_stats` GitHub release is a rolling asset that the maintainer overwrites in place rather than a versioned per-season snapshot: the files now on that release cover the 2025/26 season, and the 2018-2023 window this project needs is no longer downloadable from it at all. The historical data was recovered from `data/processed/player_match_manager.parquet`, an intermediate file saved back in task #2 that still carries the original snapshot with formation data for all 10996 rows and 0% missing. This is itself worth recording in the decision log alongside this project's other data-availability findings: a source being downloadable once is not a guarantee it stays downloadable, and pipelines that only ever re-fetch from a live upstream source with no cached original snapshot are exposed to this kind of silent scope loss.

**Why this is reported descriptively rather than as an eighth similarity dimension.** Formation is only available for the 22 club-seasons inside the manager-attributed match-level sample, never for the wider 11132-player-season population that every other dimension is z-scored against. Standardising a dimension against a population of 22 would be statistically fragile in a way the other 7 dimensions are not, so rather than quietly add a weak dimension, it is reported here as supporting qualitative evidence about each manager's tactical structure, clearly separated from the compatibility score itself.

In [12]:
import re

pmm = pd.read_parquet(DATA_PROCESSED / "player_match_manager.parquet")
pmm["own_formation"] = pmm["Home_Formation"].where(pmm["Home_Away"] == "Home", pmm["Away_Formation"])
matches = pmm[["MatchURL", "Team", "manager", "Season_End_Year", "own_formation"]].drop_duplicates(subset=["MatchURL", "Team"])
print(f"{len(matches)} distinct team-match rows across {matches['Team'].nunique()} clubs, missing formation: {matches['own_formation'].isna().sum()}")

def _back_line_size(formation):
    m = re.match(r"^(\d+)", str(formation))
    return int(m.group(1)) if m else None

matches = matches.copy()
matches["back_line"] = matches["own_formation"].apply(_back_line_size)

def _formation_profile(group):
    counts = group["own_formation"].value_counts()
    return pd.Series({
        "n_matches": len(group),
        "modal_formation": counts.index[0],
        "formation_stability": counts.iloc[0] / len(group),
        "n_distinct_formations": group["own_formation"].nunique(),
        "mean_back_line_size": group["back_line"].mean(),
    })

formation_by_club_season = matches.groupby(["manager", "Team", "Season_End_Year"]).apply(_formation_profile, include_groups=False)
formation_by_manager = matches.groupby("manager").apply(_formation_profile, include_groups=False)
print("\nPer manager, pooled across the manager's full tenure and every club:")
formation_by_manager.round(3)

769 distinct team-match rows across 6 clubs, missing formation: 0

Per manager, pooled across the manager's full tenure and every club:


,n_matches,modal_formation,formation_stability,n_distinct_formations,mean_back_line_size
manager,,,,,
Jürgen Klopp,266,4-3-3,0.865,8,3.996
Pep Guardiola,266,4-3-3,0.598,10,3.868
Roberto De Zerbi,237,4-2-3-1,0.561,15,3.810


In [13]:
formation_by_club_season.reset_index().to_csv(Path.cwd().parent / "outputs" / "tables" / "manager_formation_profile_by_club_season.csv", index=False)
formation_by_manager.reset_index().to_csv(Path.cwd().parent / "outputs" / "tables" / "manager_formation_profile.csv", index=False)
significance_summary.to_csv(Path.cwd().parent / "outputs" / "tables" / "best_fit_significance_test.csv", header=["value"])
print("saved manager_formation_profile.csv, manager_formation_profile_by_club_season.csv, best_fit_significance_test.csv")

saved manager_formation_profile.csv, manager_formation_profile_by_club_season.csv, best_fit_significance_test.csv
